In [4]:
#importing necessary libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

ModuleNotFoundError: No module named 'requests'

In [2]:
#this is the base url
BASE_URL = 'http://books.toscrape.com/catalogue/'
#Here we extract only the text content of the first page of website
START_URL = 'http://books.toscrape.com/catalogue/page-1.html'

In [3]:
#this is a function written to obtain the rating(numerical value) from the categorical value i.e 1 from 'One' for example
#here a dictionary rating is taken to show the key value pairs of numerical and categorical values
def rating(star_class):
    ratings = {"One": 1,"Two": 2,"Three": 3,"Four": 4,"Five": 5}
    for i,j in ratings.items():
        if i in star_class:
            return j
    return None

In [6]:
# this is the most important function in the program which helps to obtain information of every book
def scrape():
    # a list books is taken to put the necessary attributes as user requirements
    books = []
    # we begin with the first page of the base url
    page_number = 1
    # A loop is designed to find information of all 1000 books as required in the user story
    while len(books) < 1000:
        url = f"{BASE_URL}page-{page_number}.html"
        #to obtain response from the website we can use requests.get from the request module
        #sometimes some websites do not provide scraping oppurtunity to keep away from bot access so an additional headers line will be required but here it works without it.
        response = requests.get(url)
        #if there is no response from the website then it provides status codes from 400 generally if the website is ok then 200 is given
        if response.status_code != 200:
            print(f"Failed to retrieve page {page_number}")
            break
        #now let us extract the html type content of webpage
        soup = BeautifulSoup(response.text, 'html.parser')
        #under the article section inside the class product_pod we have the necessary book information to extract so let us navigate to it using soup.select
        book_list = soup.select('article.product_pod')
        #if there are no more books means we have come out from the class product_pod of article section so we will print no more books and come out of loop.
        if not book_list:
            print("No more books found.")
            break
        #now we navigate through items of the list and add each item
        for book in book_list:
            #the book title is a link and a h3 type and it is present inside title section
            title = book.h3.a['title']
            #select_one a function of soup returns the first matching tag as a tag object or returns none if not found
            #here price is a paragraph type and is present in price_color class so only the text part is extracted and not the currency symbol using text.strip()
            price = book.select_one('p.price_color').text.strip()
            #here the rating is accessed by star-rating class
            rating_class = book.select_one('p.star-rating')['class']
            #function rating is called
            rate = rating(rating_class)
            #availability is accessed inside instock availability class and only the text is extracted
            availability = book.select_one('p.instock.availability').text.strip()
            #the product url is a h3 type but there are various h3 elements so h3.a refers to all link elements and we require the href portion of it
            product_relative_url = book.h3.a['href']
            #each product_url are partially present so it must be joined with the base url which is done using urljoin fuction
            product_url = urljoin(BASE_URL, product_relative_url)
            #now all the attributes are appended to the books list as a key-pair value where the keys represent the name of attributes and value represents their actual value
            books.append({
                'Title': title,
                'Price': price,
                'Rating': rate,
                'Availability': 'In stock' if 'In stock' in availability else 'Out of stock',
                'Product URL': product_url
            })
        #we navigate every page so page number is increemented
        page_number +=1

    return books

In [7]:
# Scrape and save to CSV
book_data = scrape()
df = pd.DataFrame(book_data)
#remove the index assigned by to_csv operation by default
df.to_csv('books_data.csv', index=False)